In [2]:
from pyspark.sql import SparkSession
import psutil
import time

# 1. Khởi tạo Spark với config tối thiểu (để thấy khác biệt)
spark = SparkSession.builder \
    .appName("ACN_Debug") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# 2. Log system info
print("="*50)
print("SYSTEM INFO:")
print(f"RAM total: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"RAM available: {psutil.virtual_memory().available / 1024**3:.1f} GB")
print(f"CPU cores: {psutil.cpu_count()}")
print("="*50)

# 3. Đọc dữ liệu thô (giả sử file CSV)
start = time.time()
df_raw = spark.read.json("hdfs://localhost:9000/ev-project/data/bronze/ev_sessions/caltech/*/*/*")
print(f"✓ Load time: {time.time()-start:.2f}s")

# 4. Kiểm tra partition
print(f"\nPARTITION INFO:")
print(f"Number of partitions: {df_raw.rdd.getNumPartitions()}")
print(f"Sample partition size: {df_raw.rdd.glom().map(len).take(5)}")

# 5. Đếm số row (force evaluation)
start = time.time()
row_count = df_raw.count()
print(f"\n✓ Count time: {time.time()-start:.2f}s")
print(f"Total rows: {row_count:,}")

# 6. Xem schema
print("\nSCHEMA:")
df_raw.printSchema()

# 7. Lấy physical plan (quan trọng!)
print("\nPHYSICAL PLAN (first 20 lines):")
print(df_raw.queryExecution().toString()[:2000])

# 8. Lưu application ID để xem Spark UI
app_id = spark.sparkContext.applicationId
print(f"\n🔍 Spark UI: http://localhost:4040 (App ID: {app_id})")
print("👉 Hãy chụp màn hình tab 'Stages' và 'Storage'")

SYSTEM INFO:
RAM total: 7.6 GB
RAM available: 2.6 GB
CPU cores: 4


✓ Load time: 61.05s

PARTITION INFO:
Number of partitions: 34


Sample partition size: [2907, 2633, 2341, 2089, 1885]



✓ Count time: 6.16s
Total rows: 31,424

SCHEMA:
root
 |-- _batch_id: string (nullable = true)
 |-- _id: string (nullable = true)
 |-- _ingest_time: string (nullable = true)
 |-- clusterID: string (nullable = true)
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- sessionID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- spaceID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userID: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double (nullable = true)
 |    |    |-- milesRequested: long (nullable = true)
 |    |    |-- minutesAvailable: long (nullable = true)
 |    |    |-- modifiedAt: string (nullable = true)
 |    |   

AttributeError: 'DataFrame' object has no attribute 'queryExecution'

In [3]:
spark.stop()